In [7]:
import time
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

# Load data
with open("sentences.txt", "r", encoding="utf-8") as f:
    sentences = [line.strip() for line in f if line.strip()]

model = SentenceTransformer('all-MiniLM-L6-v2')
sentence_vectors = model.encode(sentences).astype('float32')



In [8]:
d = sentence_vectors.shape[1]
query = "I am a football player."
query_vector = model.encode([query]).astype('float32')

id_to_sentence = {i: s for i, s in enumerate(sentences)}

# -------------------- Flat Index --------------------
flat_index = faiss.IndexFlatL2(d)
flat_index.add(sentence_vectors)

start_time = time.perf_counter()
D_flat, I_flat = flat_index.search(query_vector, k=5)
flat_time = time.perf_counter() - start_time

print("🔍 Flat Index Results:")
for i, dist in zip(I_flat[0], D_flat[0]):
    print(f"- {id_to_sentence[i]}  (distance: {dist:.4f})")
print(f"⏱️ Flat index search time: {flat_time:.6f} seconds\n")

# -------------------- IVFPQ Index --------------------
nlist = 50
m = 8
nbits = 8

quantizer = faiss.IndexFlatL2(d)
ivfpq_index = faiss.IndexIVFPQ(quantizer, d, nlist, m, nbits)
ivfpq_index.train(sentence_vectors)
ivfpq_index.add(sentence_vectors)
ivfpq_index.nprobe = 10

start_time = time.perf_counter()
D_ivf, I_ivf = ivfpq_index.search(query_vector, k=5)
ivf_time = time.perf_counter() - start_time

print("⚡ IVFPQ Index Results:")
for i, dist in zip(I_ivf[0], D_ivf[0]):
    print(f"- {id_to_sentence[i]}  (distance: {dist:.4f})")
print(f"⏱️ IVFPQ index search time: {ivf_time:.6f} seconds\n")

# -------------------- HNSW Index --------------------
hnsw_index = faiss.IndexHNSWFlat(d, 32)  # M = 32
hnsw_index.hnsw.efSearch = 64
hnsw_index.hnsw.efConstruction = 200

hnsw_index.add(sentence_vectors)

start_time = time.perf_counter()
D_hnsw, I_hnsw = hnsw_index.search(query_vector, k=5)
hnsw_time = time.perf_counter() - start_time

print("🧭 HNSW Index Results:")
for i, dist in zip(I_hnsw[0], D_hnsw[0]):
    print(f"- {id_to_sentence[i]}  (distance: {dist:.4f})")
print(f"⏱️ HNSW index search time: {hnsw_time:.6f} seconds\n")

🔍 Flat Index Results:
- position in football played by a team member  (distance: 0.9390)
- Two groups of people are playing football  (distance: 1.0285)
- Football players are on the field.  (distance: 1.0300)
- A football player is wearing black armbands  (distance: 1.0587)
- A football player kicks the ball.  (distance: 1.0663)
⏱️ Flat index search time: 0.001114 seconds

⚡ IVFPQ Index Results:
- position in football played by a team member  (distance: 0.8019)
- A football player is carrying an official past a rolling football  (distance: 0.8298)
- A football game is being watched by the crowd  (distance: 0.8304)
- The crowd is watching a football game  (distance: 0.8385)
- The crowd is watching the football at the game  (distance: 0.8385)
⏱️ IVFPQ index search time: 0.000235 seconds

🧭 HNSW Index Results:
- position in football played by a team member  (distance: 0.9390)
- Two groups of people are playing football  (distance: 1.0285)
- Football players are on the field.  (distance: 

In [13]:
print("📊 Comparison Summary:")
print(f"Flat Search Time   : {flat_time:.6f} s")
print(f"IVFPQ Search Time  : {ivf_time:.6f} s")
print(f"HNSW Search Time   : {hnsw_time:.6f} s")

# Avoid division by zero
ivf_speedup = flat_time / ivf_time if ivf_time > 0 else float('inf')
hnsw_speedup = flat_time / hnsw_time if hnsw_time > 0 else float('inf')

print(f"IVFPQ Speedup      : {ivf_speedup:.2f}x")
print(f"HNSW Speedup       : {hnsw_speedup:.2f}x")

📊 Comparison Summary:
Flat Search Time   : 0.001114 s
IVFPQ Search Time  : 0.000235 s
HNSW Search Time   : 0.000352 s
IVFPQ Speedup      : 4.74x
HNSW Speedup       : 3.16x


In [14]:
def recall_at_k(true_indices, pred_indices, k):
    true_set = set(true_indices[0][:k])
    pred_set = set(pred_indices[0][:k])
    intersection = true_set.intersection(pred_set)
    return len(intersection) / k

recall_ivf = recall_at_k(I_flat, I_ivf, k=5)
recall_hnsw = recall_at_k(I_flat, I_hnsw, k=5)
print("📈 Evaluation Summary (Recall@5):")
print(f"IVFPQ Recall@5     : {recall_ivf:.2f}")
print(f"HNSW  Recall@5     : {recall_hnsw:.2f}")

📈 Evaluation Summary (Recall@5):
IVFPQ Recall@5     : 0.20
HNSW  Recall@5     : 1.00


In [15]:
def precision_at_k(true_indices, pred_indices, k):
    true_set = set(true_indices[0][:k])
    pred_set = set(pred_indices[0][:k])
    intersection = true_set.intersection(pred_set)
    return len(intersection) / k

precision_ivf = precision_at_k(I_flat, I_ivf, k=5)
precision_hnsw = precision_at_k(I_flat, I_hnsw, k=5)

print(f"IVFPQ Precision@5 : {precision_ivf:.2f}")
print(f"HNSW Precision@5  : {precision_hnsw:.2f}")


IVFPQ Precision@5 : 0.20
HNSW Precision@5  : 1.00


In [12]:
def f1_at_k(true_indices, pred_indices, k):
    p = precision_at_k(true_indices, pred_indices, k)
    r = recall_at_k(true_indices, pred_indices, k)
    if p + r == 0:
        return 0.0
    return 2 * (p * r) / (p + r)
f1_ivf = f1_at_k(I_flat, I_ivf, k=5)
f1_hnsw = f1_at_k(I_flat, I_hnsw, k=5)

print(f"IVFPQ F1@5 : {f1_ivf:.2f}")
print(f"HNSW  F1@5 : {f1_hnsw:.2f}")


IVFPQ F1@5 : 0.20
HNSW  F1@5 : 1.00


In [20]:
# -------------------- Evaluation Metrics --------------------
def precision_at_k(true_indices, pred_indices, k):
    true_set = set(true_indices[0][:k])
    pred_set = set(pred_indices[0][:k])
    return len(true_set & pred_set) / k

def recall_at_k(true_indices, pred_indices, k):
    true_set = set(true_indices[0][:k])
    pred_set = set(pred_indices[0][:k])
    return len(true_set & pred_set) / k

def f1_at_k(true_indices, pred_indices, k):
    p = precision_at_k(true_indices, pred_indices, k)
    r = recall_at_k(true_indices, pred_indices, k)
    return 2 * p * r / (p + r) if p + r > 0 else 0.0



# Compute all metrics for IVFPQ
recall_ivf = recall_at_k(I_flat, I_ivf, 5)
precision_ivf = precision_at_k(I_flat, I_ivf, 5)
f1_ivf = f1_at_k(I_flat, I_ivf, 5)


# Compute all metrics for HNSW
recall_hnsw = recall_at_k(I_flat, I_hnsw, 5)
precision_hnsw = precision_at_k(I_flat, I_hnsw, 5)
f1_hnsw = f1_at_k(I_flat, I_hnsw, 5)


# -------------------- Summary --------------------
print("📊 Comparison Summary:")
print(f"Flat Search Time   : {flat_time:.6f} s")
print(f"IVFPQ Search Time  : {ivf_time:.6f} s")
print(f"HNSW Search Time   : {hnsw_time:.6f} s")

ivf_speedup = flat_time / ivf_time if ivf_time > 0 else float('inf')
hnsw_speedup = flat_time / hnsw_time if hnsw_time > 0 else float('inf')

print(f"IVFPQ Speedup      : {ivf_speedup:.2f}x")
print(f"HNSW Speedup       : {hnsw_speedup:.2f}x")

print("\n📈 Evaluation Summary (Top-5):")
print(f"{'Metric':<18} | {'IVFPQ':>6} | {'HNSW':>6}")
print("-" * 38)
print(f"{'Recall@5':<18} | {recall_ivf:.2f} | {recall_hnsw:.2f}")
print(f"{'Precision@5':<18} | {precision_ivf:.2f} | {precision_hnsw:.2f}")
print(f"{'F1@5':<18} | {f1_ivf:.2f} | {f1_hnsw:.2f}")


📊 Comparison Summary:
Flat Search Time   : 0.001114 s
IVFPQ Search Time  : 0.000235 s
HNSW Search Time   : 0.000352 s
IVFPQ Speedup      : 4.74x
HNSW Speedup       : 3.16x

📈 Evaluation Summary (Top-5):
Metric             |  IVFPQ |   HNSW
--------------------------------------
Recall@5           | 0.20 | 1.00
Precision@5        | 0.20 | 1.00
F1@5               | 0.20 | 1.00


In [32]:

# Normalize for cosine similarity
sentence_vectors = sentence_vectors / np.linalg.norm(sentence_vectors, axis=1, keepdims=True)

# Define hardcoded queries
query_texts = [
    "a man is swimming in the ocean",
    "children are playing in the park",
    "a cat sits on the sofa",
    "an airplane is flying in the sky",
    "people are eating in a restaurant"
]
query_vectors = model.encode(query_texts).astype('float32')
query_vectors = query_vectors / np.linalg.norm(query_vectors, axis=1, keepdims=True)

# Build indexes
d = sentence_vectors.shape[1]
k = 5

flat_index = faiss.IndexFlatIP(d)
flat_index.add(sentence_vectors)

nlist = 50
m = 8
nbits = 8
quantizer = faiss.IndexFlatIP(d)
ivfpq_index = faiss.IndexIVFPQ(quantizer, d, nlist, m, nbits)
ivfpq_index.train(sentence_vectors)
ivfpq_index.add(sentence_vectors)
ivfpq_index.nprobe = 10

hnsw_index = faiss.IndexHNSWFlat(d, 32)
hnsw_index.hnsw.efSearch = 64
hnsw_index.hnsw.efConstruction = 200
hnsw_index.add(sentence_vectors)

# Evaluation metrics
def precision_at_k(true, pred, k):
    return len(set(true[:k]) & set(pred[:k])) / k

def recall_at_k(true, pred, k):
    return len(set(true[:k]) & set(pred[:k])) / k

def f1_at_k(true, pred, k):
    p = precision_at_k(true, pred, k)
    r = recall_at_k(true, pred, k)
    return 2 * p * r / (p + r) if p + r > 0 else 0.0

def average_precision_at_k(true, pred, k):
    hits = 0
    score = 0.0
    true_set = set(true[:k])
    for i, idx in enumerate(pred[:k]):
        if idx in true_set:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(true_set), k) if true_set else 0.0

def mean_reciprocal_rank(true, pred, k):
    true_set = set(true[:k])
    for i, idx in enumerate(pred[:k]):
        if idx in true_set:
            return 1 / (i + 1)
    return 0.0

def dcg_at_k(relevance, k):
    return sum((2 ** rel - 1) / np.log2(i + 2) for i, rel in enumerate(relevance[:k]))

def ndcg_at_k(true, pred, k):
    true_set = set(true[:k])
    relevance = [1 if idx in true_set else 0 for idx in pred[:k]]
    ideal_relevance = sorted(relevance, reverse=True)
    dcg = dcg_at_k(relevance, k)
    idcg = dcg_at_k(ideal_relevance, k)
    return dcg / idcg if idcg > 0 else 0.0

# Initialize metric storage
metrics = ["recall", "precision", "f1", "map", "mrr", "ndcg"]
results_ivf = {m: 0.0 for m in metrics}
results_hnsw = {m: 0.0 for m in metrics}
time_ivf = 0.0
time_hnsw = 0.0
time_flat = 0.0

# Evaluate over queries
for idx, query_vector in enumerate(query_vectors):
    print(f"\n🔍 Query: {query_texts[idx]}")

    # Flat search
    start = time.perf_counter()
    true_ids = flat_index.search(query_vector.reshape(1, -1), k)[1][0]
    time_flat += time.perf_counter() - start

    # IVFPQ search
    start = time.perf_counter()
    ivf_ids = ivfpq_index.search(query_vector.reshape(1, -1), k)[1][0]
    time_ivf += time.perf_counter() - start

    # HNSW search
    start = time.perf_counter()
    hnsw_ids = hnsw_index.search(query_vector.reshape(1, -1), k)[1][0]
    time_hnsw += time.perf_counter() - start

    # Print result sentences
    print("Flat Index:")
    for i in true_ids:
        print(f"- {sentences[i]}")
    print("IVFPQ Index:")
    for i in ivf_ids:
        print(f"- {sentences[i]}")
    print("HNSW Index:")
    for i in hnsw_ids:
        print(f"- {sentences[i]}")

    # Metrics for IVFPQ
    results_ivf["recall"] += recall_at_k(true_ids, ivf_ids, k)
    results_ivf["precision"] += precision_at_k(true_ids, ivf_ids, k)
    results_ivf["f1"] += f1_at_k(true_ids, ivf_ids, k)
    results_ivf["map"] += average_precision_at_k(true_ids, ivf_ids, k)
    results_ivf["mrr"] += mean_reciprocal_rank(true_ids, ivf_ids, k)
    results_ivf["ndcg"] += ndcg_at_k(true_ids, ivf_ids, k)

    # Metrics for HNSW
    results_hnsw["recall"] += recall_at_k(true_ids, hnsw_ids, k)
    results_hnsw["precision"] += precision_at_k(true_ids, hnsw_ids, k)
    results_hnsw["f1"] += f1_at_k(true_ids, hnsw_ids, k)
    results_hnsw["map"] += average_precision_at_k(true_ids, hnsw_ids, k)
    results_hnsw["mrr"] += mean_reciprocal_rank(true_ids, hnsw_ids, k)
    results_hnsw["ndcg"] += ndcg_at_k(true_ids, hnsw_ids, k)

# Compute averages
n = len(query_vectors)
for m in metrics:
    results_ivf[m] /= n
    results_hnsw[m] /= n

time_ivf /= n
time_hnsw /= n
time_flat /= n

with open("search_results.txt", "w", encoding="utf-8") as out:
    for idx, query_vector in enumerate(query_vectors):
        out.write(f"\n🔍 Query: {query_texts[idx]}\n")

        out.write("Flat Index:\n")
        for i in true_ids: out.write(f"- {sentences[i]}\n")

        out.write("IVFPQ Index:\n")
        for i in ivf_ids: out.write(f"- {sentences[i]}\n")

        out.write("HNSW Index:\n")
        for i in hnsw_ids: out.write(f"- {sentences[i]}\n")

        for metric, func in zip(metrics, [recall_at_k, precision_at_k, f1_at_k, average_precision_at_k, mean_reciprocal_rank, ndcg_at_k]):
            results_ivf[metric] += func(true_ids, ivf_ids, k)
            results_hnsw[metric] += func(true_ids, hnsw_ids, k)

    n = len(query_vectors)
    for m in metrics:
        results_ivf[m] /= n
        results_hnsw[m] /= n



    out.write("\n📈 Evaluation Summary on hardcoded queries\n")
    out.write(f"{'Metric':<10} | {'IVFPQ':>6} | {'HNSW':>6}\n")
    out.write("-" * 30 + "\n")
    for m in metrics:
        out.write(f"{m.upper():<10} | {results_ivf[m]:6.2f} | {results_hnsw[m]:6.2f}\n")



# Print evaluation summary
print("\n📈 Evaluation Summary on hardcoded queries")
print(f"{'Metric':<10} | {'IVFPQ':>6} | {'HNSW':>6}")
print("-" * 30)
for m in metrics:
    print(f"{m.upper():<10} | {results_ivf[m]:6.2f} | {results_hnsw[m]:6.2f}")





🔍 Query: a man is swimming in the ocean
Flat Index:
- A man is wind sailing in the ocean.
- The man is swimming in a body of water near a waterfall
- A man is surfing a large wave in the ocean.
- The man is going into the water
- The man is going into the water.
IVFPQ Index:
- A fish is swimming
- A person wearing scuba gear is swimming underwater water.
- A man is coming out of the water
- A man is wind sailing in the ocean.
- The man is swimming in a body of water near a waterfall
HNSW Index:
- A man is wind sailing in the ocean.
- The man is swimming in a body of water near a waterfall
- A man is surfing a large wave in the ocean.
- The man is going into the water
- The man is going into the water.

🔍 Query: children are playing in the park
Flat Index:
- Kids playing ball in the park.
- Two children are playing soccer in the park
- Some children are playing on a playground
- Some kids are playing on a playground
- The young kids are posing with a green soccer ball in a park
IVFPQ I

In [41]:
sentence_vectors = sentence_vectors / np.linalg.norm(sentence_vectors, axis=1, keepdims=True)

# Hardcoded queries
query_texts = [
    "a man is swimming in the ocean",
    "children are playing in the park",
    "a cat sits on the sofa",
    "an airplane is flying in the sky",
    "people are eating in a restaurant"
]
query_vectors = model.encode(query_texts).astype('float32')
query_vectors = query_vectors / np.linalg.norm(query_vectors, axis=1, keepdims=True)

# FAISS Indexes
d = sentence_vectors.shape[1]
k = 10

flat_index = faiss.IndexFlatIP(d)
flat_index.add(sentence_vectors)

nlist = 50
m = 8
nbits = 8
quantizer = faiss.IndexFlatIP(d)
ivfpq_index = faiss.IndexIVFPQ(quantizer, d, nlist, m, nbits)
ivfpq_index.train(sentence_vectors)
ivfpq_index.add(sentence_vectors)
ivfpq_index.nprobe = 10

hnsw_index = faiss.IndexHNSWFlat(d, 32)
hnsw_index.hnsw.efSearch = 64
hnsw_index.hnsw.efConstruction = 200
hnsw_index.add(sentence_vectors)

# Metrics
def precision_at_k(true, pred, k): return len(set(true[:k]) & set(pred[:k])) / k
def recall_at_k(true, pred, k): return len(set(true[:k]) & set(pred[:k])) / k
def f1_at_k(true, pred, k):
    p = precision_at_k(true, pred, k)
    r = recall_at_k(true, pred, k)
    return 2 * p * r / (p + r) if p + r > 0 else 0.0
def average_precision_at_k(true, pred, k):
    hits, score = 0, 0.0
    true_set = set(true[:k])
    for i, idx in enumerate(pred[:k]):
        if idx in true_set:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(true_set), k) if true_set else 0.0
def mean_reciprocal_rank(true, pred, k):
    true_set = set(true[:k])
    for i, idx in enumerate(pred[:k]):
        if idx in true_set:
            return 1 / (i + 1)
    return 0.0
def dcg_at_k(relevance, k): return sum((2 ** rel - 1) / np.log2(i + 2) for i, rel in enumerate(relevance[:k]))
def ndcg_at_k(true, pred, k):
    true_set = set(true[:k])
    relevance = [1 if idx in true_set else 0 for idx in pred[:k]]
    ideal_relevance = sorted(relevance, reverse=True)
    dcg = dcg_at_k(relevance, k)
    idcg = dcg_at_k(ideal_relevance, k)
    return dcg / idcg if idcg > 0 else 0.0

metrics = ["recall", "precision", "f1", "map", "mrr", "ndcg"]
results_ivf = {m: 0.0 for m in metrics}
results_hnsw = {m: 0.0 for m in metrics}

flat_times, ivf_times, hnsw_times = [], [], []

with open("search_results2.txt", "w", encoding="utf-8") as out:
    for idx, query_vector in enumerate(query_vectors):
        out.write(f"\n🔍 Query: {query_texts[idx]}\n")

        start = time.perf_counter()
        true_ids = flat_index.search(query_vector.reshape(1, -1), k)[1][0]
        flat_times.append(time.perf_counter() - start)

        start = time.perf_counter()
        ivf_ids = ivfpq_index.search(query_vector.reshape(1, -1), k)[1][0]
        ivf_times.append(time.perf_counter() - start)

        start = time.perf_counter()
        hnsw_ids = hnsw_index.search(query_vector.reshape(1, -1), k)[1][0]
        hnsw_times.append(time.perf_counter() - start)

        out.write("Flat Index:\n")
        for i in true_ids: out.write(f"- {sentences[i]}\n")

        out.write("IVFPQ Index:\n")
        for i in ivf_ids: out.write(f"- {sentences[i]}\n")

        out.write("HNSW Index:\n")
        for i in hnsw_ids: out.write(f"- {sentences[i]}\n")

        for metric, func in zip(metrics, [recall_at_k, precision_at_k, f1_at_k, average_precision_at_k, mean_reciprocal_rank, ndcg_at_k]):
            results_ivf[metric] += func(true_ids, ivf_ids, k)
            results_hnsw[metric] += func(true_ids, hnsw_ids, k)

    n = len(query_vectors)
    for m in metrics:
        results_ivf[m] /= n
        results_hnsw[m] /= n

    avg_flat_time = sum(flat_times) / n
    avg_ivf_time = sum(ivf_times) / n
    avg_hnsw_time = sum(hnsw_times) / n

    out.write("\n📈 Evaluation Summary on hardcoded queries\n")
    out.write(f"{'Metric':<10} | {'IVFPQ':>6} | {'HNSW':>6}\n")
    out.write("-" * 30 + "\n")
    for m in metrics:
        out.write(f"{m.upper():<10} | {results_ivf[m]:6.2f} | {results_hnsw[m]:6.2f}\n")

    out.write("\n⏱️ Average Search Time (seconds per query)\n")
    out.write(f"{'Flat':<10}: {avg_flat_time:.6f}\n")
    out.write(f"{'IVFPQ':<10}: {avg_ivf_time:.6f}\n")
    out.write(f"{'HNSW':<10}: {avg_hnsw_time:.6f}\n")
    # Avoid division by zero
    ivf_speedup = avg_flat_time / avg_ivf_time if avg_ivf_time > 0 else float('inf')
    hnsw_speedup = avg_flat_time / avg_hnsw_time if avg_hnsw_time > 0 else float('inf')

    out.write(f"\n⚡ Speedup Comparison:\n")
    out.write(f"IVFPQ Speedup      : {ivf_speedup:.2f}x\n")
    out.write(f"HNSW Speedup       : {hnsw_speedup:.2f}x\n")
